# Atividade Spark SQL

Considerando o dataset detalhado a seguir, extraia o conjunto de informações solicitadas.

### Dataset dados da BOVESPA

- Dados relativos a bovespa, a bolsa de valores no Brasil
- ~1.3GB
- 8.1M de instâncias

|  # | Nome do campo                                                  | Descrição                                                                               |
| -: | -------------------------------------------------------------- | --------------------------------------------------------------------------------------- |
|  0 | RegisterType                                                   | Fixo '1'                                                                                |
|  1 | TradingDate                                                    | Data do pregão                                                                          |
|  2 | BDICode                                                        | Utilizado para classificar os papéis na emissão do boletim diário de informações        |
|  3 | NegociationCode                                                | Código de negociação do papel                                                           |
|  4 | MarketType                                                     | Código do mercado em que o papel está cadastrado                                        |
|  5 | TradeName                                                      | Nome resumido da empresa emissora do papel                                              |
|  6 | Specification                                                  | Especificação do papel                                                                  |
|  7 | ForwardMarketTermInDays                                        | Prazo em dias do mercado a termo                                                        |
|  8 | Currency                                                       | Moeda de referência                                                                     |
|  9 | OpeningPrice                                                   | Preço de abertura do papel no pregão                                                    |
| 10 | MaxPrice                                                       | Preço máximo do papel no pregão                                                         |
| 11 | MinPrice                                                       | Preço mínimo do papel no pregão                                                         |
| 12 | MeanPrice                                                      | Preço médio do papel no pregão                                                          |
| 13 | LastTradePrice                                                 | Preço do último negócio do papel no pregão                                              |
| 14 | BestPurshaseOrderPrice                                         | Preço da melhor oferta de compra do papel no mercado                                    |
| 15 | BestPurshaseSalePrice                                          | Preço da melhor oferta de venda do papel no mercado                                     |
| 16 | NumborOfTrades                                                 | Número de negócios efetuados com o papel no pregão                                      |
| 17 | NumberOfTradedStocks                                           | Quantidade total de títulos negociados neste papel                                      |
| 18 | VolumeOfTradedStocks                                           | Volume total de títulos negociados neste papel                                          |
| 19 | PriceForOptionsMarketOrSecondaryTermMarket                     | Preço de exercício para o mercado de opções ou valor do contrato para o mercado a termo |
| 20 | PriceCorrectionsForOptionsMarketOrSecondaryTermMarket          | Indicador de correção de preços de exercício ou valores de contrato                     |
| 21 | DueDateForOptionsMarketOrSecondaryTermMarket                   | Data de vencimento para os mercados de opções                                           |
| 22 | FactorOfPaperQuotatuion                                        | Fator de cotação do papel                                                               |
| 23 | PointsInPriceForOptionsMarketReferencedInDollarOrSecondaryTerm | Preço de exercício em pontos para opções referenciadas em dólar                         |
| 24 | ISINOrInternCode                                               | Código do papel no sistema ISIN                                                         |
| 25 | DistributionNumber                                             | Número de distribuição do papel                                                         |


Informações a serem extraídas:

1. Quantidade de dias com operações da PETR4 (NegociationCode)
2. Maior valor (MaxPrice) histórico por ação (NegociationCode)
3. Maior valor (MaxPrice) histórico da PETR4 (NegociationCode)
4. Dia ('TradingDate') com a maior quantidade de papeis (NegociationCode) operados
5. Dia ('TradingDate') da semana com a maior quantidade de papeis (NegociationCode) operados
6. Maior lucro histórico de um papel (NegociationCode) na bovespa (MaxPrice - OpeningPrice)
7. Maior prejuizo histórico de um papel (NegociationCode) na bovespa (OpeningPrice - LastTradePrice)
8. Moeda (Currency) com mais operações
9. Papel (NegociationCode) operado em CZ (Currency) com maior quantidade de operações
10. Papel (NegociationCode) operado em CZ (Currency) com maior valor médio das operações (MeanPrice)
11. Media do preço médio (MeanPrice), mínimo (MinPrice) e máximo (MaxPrice) anual (TradingDate) das ações da PETR4 (NegociationCode)
12. Preço médio (MeanPrice) anual (TradingDate) das ações da PETR4 (NegociationCode)
13. Preço médio (MeanPrice) anual (TradingDate) das 10 ações (NegociationCode) com mais operações na bovespa
14. Desvio Padrão anual do preço médio (MeanPrice) da ação da PETR4 (NegociationCode)
15. Desvio Padrão anual do preço médio (MeanPrice) das 10 ações (NegociationCode) com mais operações na bovespa
16. Preço médio (MeanPrice) anual (TradingDate) das ações (NegociationCode) com a maior quantidade de operações de acordo com a moeda (Currency)


**Dicas:**
- *Crie uma célula (Insert -> Insert Cell Below) para cada informação solicitada*

In [1]:
#instala pyspark no google colab
# (instala apenas o que faltar; o notebook não depende de nenhum outro arquivo ou pasta)
import importlib.util, os
if importlib.util.find_spec('gdown') is None:
    !pip --quiet install --upgrade --no-cache-dir gdown
if importlib.util.find_spec('pyspark') is None:
    !pip install pyspark==3.5.9

In [2]:
# abre sessão no spark sql
from pyspark.sql import SparkSession

sc = SparkSession \
    .builder \
    .master('local[*]') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
sc.sparkContext.setLogLevel('ERROR')
print('Spark', sc.version)

Spark 3.5.6


In [3]:
#Efetua download do arquivo bovespa
# baixa a base do link do professor para a pasta de execução (no Colab, /content), se ainda não existir
ARQUIVO = 'bovespa.csv'
if not os.path.exists(ARQUIVO):
    !gdown 1FbkYKnij5N6A2P3VLgfMLYhoPSb4_wur
CAMINHO = 'file://' + os.path.abspath(ARQUIVO)
print(CAMINHO)

In [4]:
df = sc.read \
    .option('delimiter', ',') \
    .option('header', 'true') \
    .option('inferschema', 'true') \
    .csv(CAMINHO)

## Preparação dos dados

Cada item é resolvido de duas formas, como nos slides de Spark SQL: primeiro com a **API de DataFrame** (`select`, `where`, `groupBy`, `agg`, `orderBy`) e, na célula seguinte, com a **consulta SQL equivalente** sobre a visão temporária `bovespa` (`sc.sql(...)`). As duas passam pelo mesmo otimizador (Catalyst) e geram o mesmo plano físico, então os resultados são idênticos.

Antes das consultas:

1. conferimos o schema inferido e o tamanho da base;
2. criamos as colunas derivadas `data` (a partir de `TradingDate`, que vem como inteiro `aaaammdd`), `ano` e `dia_semana`;
3. mantemos só as colunas usadas e fazemos `cache()`, porque o DataFrame é consultado mais de 30 vezes.

In [5]:
df.printSchema()
print('Registros:', df.count())

root
 |-- RegisterType: integer (nullable = true)
 |-- TradingDate: integer (nullable = true)
 |-- BDICode: double (nullable = true)
 |-- NegociationCode: string (nullable = true)
 |-- MarketType: integer (nullable = true)
 |-- TradeName: string (nullable = true)
 |-- Specification: string (nullable = true)
 |-- ForwardMarketTermInDays: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- OpeningPrice: double (nullable = true)
 |-- MaxPrice: double (nullable = true)
 |-- MinPrice: double (nullable = true)
 |-- MeanPrice: double (nullable = true)
 |-- LastTradePrice: double (nullable = true)
 |-- BestPurshaseOrderPrice: double (nullable = true)
 |-- BestPurshaseSalePrice: double (nullable = true)
 |-- NumborOfTrades: double (nullable = true)
 |-- NumberOfTradedStocks: double (nullable = true)
 |-- VolumeOfTradedStocks: double (nullable = true)
 |-- PriceForOptionsMarketOrSecondaryTermMarket: double (nullable = true)
 |-- PriceCorrectionsForOptionsMarketOrSecondaryTermMa

Registros: 8125691


In [6]:
import pyspark.sql.functions as func

DIAS = {1: 'Domingo', 2: 'Segunda', 3: 'Terça', 4: 'Quarta', 5: 'Quinta', 6: 'Sexta', 7: 'Sábado'}
mapa_dias = func.create_map([func.lit(x) for kv in DIAS.items() for x in kv])

bolsa = df.select('TradingDate', 'NegociationCode', 'TradeName', 'MarketType', 'Currency',
                  'OpeningPrice', 'MaxPrice', 'MinPrice', 'MeanPrice', 'LastTradePrice',
                  'NumborOfTrades')\
          .withColumn('NegociationCode', func.trim(func.col('NegociationCode')))\
          .withColumn('data', func.to_date(func.col('TradingDate').cast('string'), 'yyyyMMdd'))\
          .withColumn('ano', func.year('data'))\
          .withColumn('dia_semana', mapa_dias[func.dayofweek('data')])\
          .cache()

bolsa.createOrReplaceTempView('bovespa')
print('Registros em cache:', bolsa.count())
bolsa.select(func.min('data').alias('primeiro_pregao'), func.max('data').alias('ultimo_pregao'),
             func.countDistinct('data').alias('pregoes'),
             func.countDistinct('NegociationCode').alias('papeis')).show()

Registros em cache: 8125691


+---------------+-------------+-------+------+
|primeiro_pregao|ultimo_pregao|pregoes|papeis|
+---------------+-------------+-------+------+
|     1986-01-02|   2019-12-30|   8385|183127|
+---------------+-------------+-------+------+



In [7]:
bolsa.select('data', 'dia_semana', 'NegociationCode', 'TradeName', 'Currency',
             'OpeningPrice', 'MaxPrice', 'MinPrice', 'MeanPrice', 'LastTradePrice').show(5)

+----------+----------+---------------+----------+--------+------------+--------+--------+---------+--------------+
|      data|dia_semana|NegociationCode| TradeName|Currency|OpeningPrice|MaxPrice|MinPrice|MeanPrice|LastTradePrice|
+----------+----------+---------------+----------+--------+------------+--------+--------+---------+--------------+
|1986-01-02|    Quinta|          ACE 2|   ACESITA|     CR$|       430.0|   430.0|   430.0|    430.0|         430.0|
|1986-01-02|    Quinta|          AVI 2| ACOS VILL|     CR$|      2300.0|  2500.0|  2250.0|   2356.0|        2499.0|
|1986-01-02|    Quinta|          CRA 2|ADUBOS CRA|     CR$|       150.0|   150.0|   150.0|    150.0|         150.0|
|1986-01-02|    Quinta|         AGR 11|    AGRALE|     CR$|    115100.0|115100.0|115100.0| 115100.0|      115100.0|
|1986-01-02|    Quinta|          SAG 2| AGROCERES|     CR$|      1750.0|  1850.0|  1700.0|   1779.0|        1800.0|
+----------+----------+---------------+----------+--------+------------+

**Definições adotadas:**

- **ação / papel**: valor de `NegociationCode` (sem espaços nas pontas). A base traz todos os mercados (à vista, fracionário, opções, termo), então "papel" inclui opções e outros instrumentos, como no enunciado;
- **operação**: um registro da base, isto é, um papel negociado em um pregão (é a contagem usada nos itens 8, 9, 13, 15 e 16);
- **CZ**: moeda `CZ$` (cruzado, de março de 1986 a janeiro de 1989);
- **anual**: agrupamento pelo ano de `TradingDate`.

**Atenção à unidade dos preços.** A base vem do arquivo histórico da B3 (COTAHIST), em que os preços têm duas casas decimais implícitas: `2724.5` é a média da PETR4 em 2019 e corresponde a R$ 27,24. Além disso, cada valor está na moeda da época (CR$, CZ$, NCZ$, R$) e no fator de cotação do papel (`FactorOfPaperQuotatuion`): até 2000 a PETR4 era cotada por lote de mil ações (fator 1000) e a partir de então por ação (fator 1). Os números abaixo são os valores da base, sem conversão; as observações em cada resposta indicam onde essas diferenças de unidade afetam a leitura.

In [8]:
# 1. Quantidade de dias com operações da PETR4 (NegociationCode)
dias_petr4 = bolsa.where(func.col('NegociationCode') == 'PETR4')\
                  .select('data').distinct().count()
print('Dias com operações da PETR4:', dias_petr4)

Dias com operações da PETR4: 5391


In [9]:
sc.sql("""
    SELECT COUNT(DISTINCT data) AS dias_com_operacao,
           MIN(data) AS primeiro_dia, MAX(data) AS ultimo_dia
    FROM bovespa
    WHERE NegociationCode = 'PETR4'
""").show()

+-----------------+------------+----------+
|dias_com_operacao|primeiro_dia|ultimo_dia|
+-----------------+------------+----------+
|             5391|  1998-03-16|2019-12-30|
+-----------------+------------+----------+



**Resposta.** A **PETR4 teve operações em 5.391 dias** (pregões), de 16/03/1998 a 30/12/2019.

In [10]:
# 2. Maior valor (MaxPrice) histórico por ação (NegociationCode)
max_por_acao = bolsa.groupBy('NegociationCode')\
                    .agg(func.max('MaxPrice').alias('maior_MaxPrice'))\
                    .orderBy(func.col('maior_MaxPrice').desc())
print('Papéis distintos:', max_por_acao.count())
max_por_acao.show(15)

Papéis distintos: 183127


+---------------+---------------+
|NegociationCode| maior_MaxPrice|
+---------------+---------------+
|          NCT 5|  2.74484259E10|
|          MBA 5|1.2408903293E10|
|          BMG 4|          3.5E9|
|          BMG 3|          3.5E9|
|          BMG 2|          3.5E9|
|          ANT 3|       1.2669E9|
|          RHE 4|          1.1E9|
|          RHE 3|         1.05E9|
|          SLE 4|   9.99999999E8|
|          ARN 2|          9.5E8|
|          ARN 3|          9.5E8|
|          ARN 4|          8.5E8|
|          SLE 2|          8.0E8|
|        BVSP11L|         7.64E8|
|          ANT 4|          7.2E8|
+---------------+---------------+
only showing top 15 rows



In [11]:
sc.sql("""
    SELECT NegociationCode, MAX(MaxPrice) AS maior_MaxPrice
    FROM bovespa
    GROUP BY NegociationCode
    ORDER BY NegociationCode
""").show(10)

+---------------+--------------+
|NegociationCode|maior_MaxPrice|
+---------------+--------------+
|         A1LX34|       22597.0|
|        A1LX34F|       22597.0|
|        AALC11B|        3717.0|
|         AALC34|       10058.0|
|        AALC34F|       10058.0|
|         AALL34|       18737.0|
|        AALL34F|       19389.0|
|          AALR3|        1976.0|
|         AALR3F|        1999.0|
|         AALR3T|        1956.0|
+---------------+--------------+
only showing top 10 rows



**Resposta.** A base tem **183.127 papéis distintos**; a tabela acima lista os maiores `MaxPrice` históricos, liderados por **NCT 5 (2,74 × 10¹⁰)** e **MBA 5 (1,24 × 10¹⁰)**. A célula SQL mostra o mesmo cálculo em ordem alfabética, que é o formato "um valor por ação" pedido. Os maiores valores vêm todos da época de inflação alta (CR$/NCZ$), por isso não são comparáveis aos preços em real.

In [12]:
# 3. Maior valor (MaxPrice) histórico da PETR4 (NegociationCode)
bolsa.where(func.col('NegociationCode') == 'PETR4')\
     .agg(func.max('MaxPrice').alias('maior_MaxPrice_PETR4')).show()

+--------------------+
|maior_MaxPrice_PETR4|
+--------------------+
|             52100.0|
+--------------------+



In [13]:
# em SQL, também recuperando o dia em que o máximo ocorreu
sc.sql("""
    SELECT data, Currency, MaxPrice
    FROM bovespa
    WHERE NegociationCode = 'PETR4'
      AND MaxPrice = (SELECT MAX(MaxPrice) FROM bovespa WHERE NegociationCode = 'PETR4')
""").show()

+----------+--------+--------+
|      data|Currency|MaxPrice|
+----------+--------+--------+
|2000-06-21|      R$| 52100.0|
+----------+--------+--------+



**Resposta.** O **maior `MaxPrice` histórico da PETR4 é 52.100**, em **21/06/2000** (moeda R$). Pela unidade da base, são R$ 521,00 por lote de mil ações, já que em 2000 a PETR4 ainda era cotada com fator 1000.

In [14]:
# 4. Dia ('TradingDate') com a maior quantidade de papeis (NegociationCode) operados
papeis_por_dia = bolsa.groupBy('data')\
                      .agg(func.countDistinct('NegociationCode').alias('papeis_operados'))\
                      .orderBy(func.col('papeis_operados').desc())
papeis_por_dia.show(5)

+----------+---------------+
|      data|papeis_operados|
+----------+---------------+
|2019-12-16|           4323|
|2019-08-19|           3965|
|2019-12-13|           3962|
|2019-11-18|           3952|
|2019-12-12|           3950|
+----------+---------------+
only showing top 5 rows



In [15]:
sc.sql("""
    SELECT data, dia_semana, COUNT(DISTINCT NegociationCode) AS papeis_operados
    FROM bovespa
    GROUP BY data, dia_semana
    ORDER BY papeis_operados DESC
    LIMIT 5
""").show()

+----------+----------+---------------+
|      data|dia_semana|papeis_operados|
+----------+----------+---------------+
|2019-12-16|   Segunda|           4323|
|2019-08-19|   Segunda|           3965|
|2019-12-13|     Sexta|           3962|
|2019-11-18|   Segunda|           3952|
|2019-12-12|    Quinta|           3950|
+----------+----------+---------------+



**Resposta.** O dia com mais papéis distintos operados foi **16/12/2019 (segunda-feira), com 4.323 papéis**. Os cinco primeiros dias do ranking são todos do segundo semestre de 2019, reflexo do crescimento do número de instrumentos listados (sobretudo opções) ao longo do tempo.

In [16]:
# 5. Dia ('TradingDate') da semana com a maior quantidade de papeis (NegociationCode) operados
# soma, em todos os pregões, dos papéis operados em cada dia da semana; e média por pregão
bolsa.groupBy('data', 'dia_semana')\
     .agg(func.countDistinct('NegociationCode').alias('papeis'))\
     .groupBy('dia_semana')\
     .agg(func.sum('papeis').alias('papeis_operados'),
          func.count('data').alias('pregoes'),
          func.round(func.avg('papeis'), 1).alias('media_por_pregao'))\
     .orderBy(func.col('papeis_operados').desc())\
     .show()

+----------+---------------+-------+----------------+
|dia_semana|papeis_operados|pregoes|media_por_pregao|
+----------+---------------+-------+----------------+
|    Quarta|        1465033|   1702|           860.8|
|   Segunda|        1447165|   1664|           869.7|
|     Terça|        1416588|   1682|           842.2|
|    Quinta|        1411882|   1667|           847.0|
|     Sexta|        1411563|   1669|           845.8|
|    Sábado|              3|      1|             3.0|
+----------+---------------+-------+----------------+



In [17]:
sc.sql("""
    SELECT dia_semana, COUNT(*) AS papeis_operados
    FROM bovespa
    GROUP BY dia_semana
    ORDER BY papeis_operados DESC
""").show()

+----------+---------------+
|dia_semana|papeis_operados|
+----------+---------------+
|    Quarta|        1688444|
|   Segunda|        1626494|
|     Terça|        1613747|
|     Sexta|        1599650|
|    Quinta|        1597353|
|    Sábado|              3|
+----------+---------------+



**Resposta.** **Quarta-feira** é o dia da semana com mais papéis operados: 1.688.444 registros (1.465.033 somando os papéis distintos de cada pregão). A diferença para os outros dias é pequena e decorre em boa parte de haver mais pregões às quartas (1.702). Pela **média por pregão**, a segunda-feira fica à frente (869,7 papéis contra 860,8 da quarta). Aparece também um único pregão em um sábado, com 3 papéis.

In [18]:
# 6. Maior lucro histórico de um papel (NegociationCode) na bovespa (MaxPrice - OpeningPrice)
lucro = bolsa.withColumn('lucro', func.col('MaxPrice') - func.col('OpeningPrice'))\
             .orderBy(func.col('lucro').desc())
lucro.select('NegociationCode', 'TradeName', 'data', 'Currency', 'OpeningPrice', 'MaxPrice', 'lucro').show(5)

+---------------+-----------+----------+--------+------------+-----------+-----------+
|NegociationCode|  TradeName|      data|Currency|OpeningPrice|   MaxPrice|      lucro|
+---------------+-----------+----------+--------+------------+-----------+-----------+
|          ANT 3| ANTARCTICA|1990-12-12|     CR$|       4.8E8|    9.211E8|    4.411E8|
|          ANT 3| ANTARCTICA|1990-11-28|     CR$|       4.2E8|      6.5E8|      2.3E8|
|          ARN 2|       ARNO|1990-12-07|     CR$|       4.7E8|      6.2E8|      1.5E8|
|          TEL 4|   TELEBRAS|1992-08-12|     CR$|      4400.0|1.0000448E8|1.0000008E8|
|          RHE 4|REN HERMANN|1989-12-28|    NCZ$|       3.0E8|      4.0E8|      1.0E8|
+---------------+-----------+----------+--------+------------+-----------+-----------+
only showing top 5 rows



In [19]:
# o mesmo cálculo restrito ao real (R$), que permite comparar valores entre pregões
sc.sql("""
    SELECT NegociationCode, TradeName, data, OpeningPrice, MaxPrice,
           MaxPrice - OpeningPrice AS lucro
    FROM bovespa
    WHERE Currency = 'R$'
    ORDER BY lucro DESC
    LIMIT 5
""").show()

+---------------+---------+----------+------------+--------+---------+
|NegociationCode|TradeName|      data|OpeningPrice|MaxPrice|    lucro|
+---------------+---------+----------+------------+--------+---------+
|        LIGH11F|    LIGHT|2005-08-09|   9950000.0|   9.0E7|  8.005E7|
|        LIGH11F|    LIGHT|2005-08-15|       3.0E7|   9.0E7|    6.0E7|
|        LIGH11F|    LIGHT|2005-08-22|      3.89E7|   5.0E7|   1.11E7|
|        LIGH11F|    LIGHT|2005-08-23|   4.50001E7|  5.35E7|8499900.0|
|        LIGH11F|    LIGHT|2005-08-31|      4.25E7|   4.8E7|5500000.0|
+---------------+---------+----------+------------+--------+---------+



**Resposta.** O maior lucro histórico (`MaxPrice - OpeningPrice`) foi da **ANT 3 (Antarctica) em 12/12/1990: 441.100.000 CR$** (abriu em 480.000.000 e chegou a 921.100.000). Como é um valor em cruzeiro, em plena hiperinflação, a célula SQL repete o cálculo **só em real**: o maior lucro é do **LIGH11F (Light, mercado fracionário) em 09/08/2005, 80.050.000** na unidade da base.

In [20]:
# 7. Maior prejuizo histórico de um papel (NegociationCode) na bovespa (OpeningPrice - LastTradePrice)
prejuizo = bolsa.withColumn('prejuizo', func.col('OpeningPrice') - func.col('LastTradePrice'))\
                .orderBy(func.col('prejuizo').desc())
prejuizo.select('NegociationCode', 'TradeName', 'data', 'Currency', 'OpeningPrice', 'LastTradePrice', 'prejuizo').show(5)

+---------------+------------+----------+--------+------------+--------------+--------+
|NegociationCode|   TradeName|      data|Currency|OpeningPrice|LastTradePrice|prejuizo|
+---------------+------------+----------+--------+------------+--------------+--------+
|          ANT 3|  ANTARCTICA|1993-07-14|     CR$|     1.165E9|        1.02E9|  1.45E8|
|          SLE 2|SARAIVA LIVR|1990-10-23|     CR$|       3.0E8|         2.0E8|   1.0E8|
|          ANT 3|  ANTARCTICA|1993-07-26|     CR$|     1.221E9|        1.17E9|   5.1E7|
|          RHE 4| REN HERMANN|1990-06-04|     CR$|       5.0E8|         4.5E8|   5.0E7|
|          ANT 3|  ANTARCTICA|1993-05-25|     CR$|       6.0E8|         5.5E8|   5.0E7|
+---------------+------------+----------+--------+------------+--------------+--------+
only showing top 5 rows



In [21]:
sc.sql("""
    SELECT NegociationCode, TradeName, data, OpeningPrice, LastTradePrice,
           OpeningPrice - LastTradePrice AS prejuizo
    FROM bovespa
    WHERE Currency = 'R$'
    ORDER BY prejuizo DESC
    LIMIT 5
""").show()

+---------------+---------+----------+------------+--------------+---------+
|NegociationCode|TradeName|      data|OpeningPrice|LastTradePrice| prejuizo|
+---------------+---------+----------+------------+--------------+---------+
|        LIGH11F|    LIGHT|2005-08-16|       4.9E7|        3.85E7|   1.05E7|
|        LIGH11F|    LIGHT|2005-08-26|       5.2E7|         4.6E7|6000000.0|
|        LIGH11F|    LIGHT|2005-08-10|       5.0E7|         4.6E7|4000000.0|
|          ARN 4|     ARNO|1994-07-14|      2.48E7|         2.1E7|3800000.0|
|        LIGH11F|    LIGHT|2005-08-29|   5.09999E7|         4.8E7|2999900.0|
+---------------+---------+----------+------------+--------------+---------+



**Resposta.** O maior prejuízo histórico (`OpeningPrice - LastTradePrice`) foi da **ANT 3 (Antarctica) em 14/07/1993: 145.000.000 CR$** (abriu em 1.165.000.000 e fechou em 1.020.000.000). **Restrito ao real**, o maior é do **LIGH11F (Light) em 16/08/2005: 10.500.000** na unidade da base.

In [22]:
# 8. Moeda (Currency) com mais operações
bolsa.groupBy('Currency').count().orderBy(func.col('count').desc()).show()

+--------+-------+
|Currency|  count|
+--------+-------+
|      R$|6995662|
|     CR$| 498562|
|     CZ$| 458842|
|    NCZ$| 172625|
+--------+-------+



In [23]:
sc.sql("""
    SELECT Currency, COUNT(*) AS operacoes, MIN(data) AS de, MAX(data) AS ate
    FROM bovespa
    GROUP BY Currency
    ORDER BY operacoes DESC
""").show()

+--------+---------+----------+----------+
|Currency|operacoes|        de|       ate|
+--------+---------+----------+----------+
|      R$|  6995662|1994-07-04|2019-12-30|
|     CR$|   498562|1986-01-02|1994-06-30|
|     CZ$|   458842|1986-03-04|1989-01-13|
|    NCZ$|   172625|1989-01-18|1990-03-13|
+--------+---------+----------+----------+



**Resposta.** O **real (R$) é a moeda com mais operações: 6.995.662 registros** (86 % da base), de 04/07/1994 a 30/12/2019. Em seguida vêm CR$ (498.562), CZ$ (458.842) e NCZ$ (172.625).

In [24]:
# 9. Papel (NegociationCode) operado em CZ (Currency) com maior quantidade de operações
bolsa.where(func.col('Currency') == 'CZ$')\
     .groupBy('NegociationCode')\
     .count()\
     .orderBy(func.col('count').desc())\
     .show(5)

+---------------+-----+
|NegociationCode|count|
+---------------+-----+
|          PMA 2| 2002|
|          PET 2| 1993|
|          SHA 2| 1755|
|          BB  2| 1751|
|          BES 2| 1749|
+---------------+-----+
only showing top 5 rows



In [25]:
sc.sql("""
    SELECT NegociationCode, FIRST(TradeName) AS empresa, COUNT(*) AS operacoes
    FROM bovespa
    WHERE Currency = 'CZ$'
    GROUP BY NegociationCode
    ORDER BY operacoes DESC
    LIMIT 5
""").show()

+---------------+------------+---------+
|NegociationCode|     empresa|operacoes|
+---------------+------------+---------+
|          PMA 2|PARANAPANEMA|     2002|
|          PET 2|   PETROBRAS|     1993|
|          SHA 2|       SHARP|     1755|
|          BB  2|      BRASIL|     1751|
|          BES 2|     BANESPA|     1749|
+---------------+------------+---------+



**Resposta.** Em cruzado (CZ$), o papel com mais operações é o **PMA 2 (Paranapanema), com 2.002 operações**, seguido de PET 2 (Petrobras, 1.993) e SHA 2 (Sharp, 1.755).

In [26]:
# 10. Papel (NegociationCode) operado em CZ (Currency) com maior valor médio das operações (MeanPrice)
bolsa.where(func.col('Currency') == 'CZ$')\
     .groupBy('NegociationCode')\
     .agg(func.mean('MeanPrice').alias('valor_medio'), func.count('*').alias('operacoes'))\
     .orderBy(func.col('valor_medio').desc())\
     .show(5)

+---------------+-----------+---------+
|NegociationCode|valor_medio|operacoes|
+---------------+-----------+---------+
|          ARN 4|  4000000.0|        1|
|          AHE 3|  3250000.0|        2|
|          ALB 5|  2600000.0|        1|
|          CCC 3|  2500000.0|        1|
|          CCC 4|  2500000.0|        1|
+---------------+-----------+---------+
only showing top 5 rows



In [27]:
sc.sql("""
    SELECT NegociationCode, FIRST(TradeName) AS empresa,
           AVG(MeanPrice) AS valor_medio, COUNT(*) AS operacoes
    FROM bovespa
    WHERE Currency = 'CZ$'
    GROUP BY NegociationCode
    ORDER BY valor_medio DESC
    LIMIT 5
""").show()

+---------------+----------+-----------+---------+
|NegociationCode|   empresa|valor_medio|operacoes|
+---------------+----------+-----------+---------+
|          ARN 4|      ARNO|  4000000.0|        1|
|          AHE 3|   ANHEMBI|  3250000.0|        2|
|          ALB 5|   ALBARUS|  2600000.0|        1|
|          CCC 3|CAM CORREA|  2500000.0|        1|
|          CCC 4|CAM CORREA|  2500000.0|        1|
+---------------+----------+-----------+---------+



**Resposta.** Em cruzado (CZ$), o maior valor médio das operações é o do **ARN 4 (Arno), com `MeanPrice` médio de 4.000.000**. É um papel com **uma única operação** no período, assim como os seguintes do ranking; a média alta reflete um negócio isolado, não um papel caro de forma recorrente.

In [28]:
# 11. Media do preço médio (MeanPrice), mínimo (MinPrice) e máximo (MaxPrice) anual (TradingDate) das ações da PETR4 (NegociationCode)
bolsa.where(func.col('NegociationCode') == 'PETR4')\
     .groupBy('ano')\
     .agg(func.round(func.mean('MeanPrice'), 2).alias('media_MeanPrice'),
          func.round(func.mean('MinPrice'), 2).alias('media_MinPrice'),
          func.round(func.mean('MaxPrice'), 2).alias('media_MaxPrice'))\
     .orderBy('ano')\
     .show(30)

+----+---------------+--------------+--------------+
| ano|media_MeanPrice|media_MinPrice|media_MaxPrice|
+----+---------------+--------------+--------------+
|1998|        20242.4|      19773.34|      20707.29|
|1999|       26273.99|      25791.16|       26727.6|
|2000|       24186.14|      23769.96|      24586.69|
|2001|         5297.7|       5216.28|       5383.42|
|2002|        4796.74|       4727.74|       4871.04|
|2003|        5613.91|        5547.0|       5681.28|
|2004|        8383.64|       8279.13|       8488.27|
|2005|        8094.47|       7993.02|       8192.24|
|2006|        4344.06|       4285.99|       4400.65|
|2007|        5442.98|       5357.36|        5522.3|
|2008|        4806.37|       4705.02|       4906.42|
|2009|        3183.65|       3141.59|       3225.02|
|2010|        2975.07|       2940.07|       3011.08|
|2011|        2369.92|       2344.15|       2396.61|
|2012|        2134.69|       2106.91|        2163.5|
|2013|        1825.31|       1800.57|       18

In [29]:
sc.sql("""
    SELECT ano,
           ROUND(AVG(MeanPrice), 2) AS media_MeanPrice,
           ROUND(AVG(MinPrice), 2)  AS media_MinPrice,
           ROUND(AVG(MaxPrice), 2)  AS media_MaxPrice
    FROM bovespa
    WHERE NegociationCode = 'PETR4'
    GROUP BY ano
    ORDER BY ano
""").show(30)

+----+---------------+--------------+--------------+
| ano|media_MeanPrice|media_MinPrice|media_MaxPrice|
+----+---------------+--------------+--------------+
|1998|        20242.4|      19773.34|      20707.29|
|1999|       26273.99|      25791.16|       26727.6|
|2000|       24186.14|      23769.96|      24586.69|
|2001|         5297.7|       5216.28|       5383.42|
|2002|        4796.74|       4727.74|       4871.04|
|2003|        5613.91|        5547.0|       5681.28|
|2004|        8383.64|       8279.13|       8488.27|
|2005|        8094.47|       7993.02|       8192.24|
|2006|        4344.06|       4285.99|       4400.65|
|2007|        5442.98|       5357.36|        5522.3|
|2008|        4806.37|       4705.02|       4906.42|
|2009|        3183.65|       3141.59|       3225.02|
|2010|        2975.07|       2940.07|       3011.08|
|2011|        2369.92|       2344.15|       2396.61|
|2012|        2134.69|       2106.91|        2163.5|
|2013|        1825.31|       1800.57|       18

**Resposta.** As médias anuais da PETR4 estão na tabela acima (1998 a 2019). Os três valores andam juntos, com a média do preço mínimo sempre um pouco abaixo e a do máximo um pouco acima da média do preço médio. Exemplo: em 2019, médias de 2.724,50 (preço médio), 2.693,62 (mínimo) e 2.755,37 (máximo), isto é, cerca de R$ 27,24, R$ 26,94 e R$ 27,55 por ação. A queda brusca de 2000 (24.186) para 2001 (5.298) é a mudança do fator de cotação, não uma desvalorização.

In [30]:
# 12. Preço médio (MeanPrice) anual (TradingDate) das ações da PETR4 (NegociationCode)
petr4_ano = bolsa.where(func.col('NegociationCode') == 'PETR4')\
                 .groupBy('ano')\
                 .agg(func.round(func.mean('MeanPrice'), 2).alias('preco_medio'))\
                 .orderBy('ano')
petr4_ano.show(30)

+----+-----------+
| ano|preco_medio|
+----+-----------+
|1998|    20242.4|
|1999|   26273.99|
|2000|   24186.14|
|2001|     5297.7|
|2002|    4796.74|
|2003|    5613.91|
|2004|    8383.64|
|2005|    8094.47|
|2006|    4344.06|
|2007|    5442.98|
|2008|    4806.37|
|2009|    3183.65|
|2010|    2975.07|
|2011|    2369.92|
|2012|    2134.69|
|2013|    1825.31|
|2014|    1669.73|
|2015|     980.43|
|2016|    1077.49|
|2017|    1459.06|
|2018|    2102.45|
|2019|     2724.5|
+----+-----------+



In [31]:
sc.sql("""
    SELECT ano, ROUND(AVG(MeanPrice), 2) AS preco_medio
    FROM bovespa
    WHERE NegociationCode = 'PETR4'
    GROUP BY ano
    ORDER BY ano
""").show(30)

+----+-----------+
| ano|preco_medio|
+----+-----------+
|1998|    20242.4|
|1999|   26273.99|
|2000|   24186.14|
|2001|     5297.7|
|2002|    4796.74|
|2003|    5613.91|
|2004|    8383.64|
|2005|    8094.47|
|2006|    4344.06|
|2007|    5442.98|
|2008|    4806.37|
|2009|    3183.65|
|2010|    2975.07|
|2011|    2369.92|
|2012|    2134.69|
|2013|    1825.31|
|2014|    1669.73|
|2015|     980.43|
|2016|    1077.49|
|2017|    1459.06|
|2018|    2102.45|
|2019|     2724.5|
+----+-----------+



**Resposta.** Preço médio anual da PETR4 (unidade da base; divida por 100 para obter reais):

| Ano | Preço médio | Ano | Preço médio | Ano | Preço médio |
|---|---|---|---|---|---|
| 1998 | 20.242,40 | 2006 | 4.344,06 | 2014 | 1.669,73 |
| 1999 | 26.273,99 | 2007 | 5.442,98 | 2015 | 980,43 |
| 2000 | 24.186,14 | 2008 | 4.806,37 | 2016 | 1.077,49 |
| 2001 | 5.297,70 | 2009 | 3.183,65 | 2017 | 1.459,06 |
| 2002 | 4.796,74 | 2010 | 2.975,07 | 2018 | 2.102,45 |
| 2003 | 5.613,91 | 2011 | 2.369,92 | 2019 | 2.724,50 |
| 2004 | 8.383,64 | 2012 | 2.134,69 | | |
| 2005 | 8.094,47 | 2013 | 1.825,31 | | |

O menor preço médio anual foi o de 2015 (R$ 9,80), ano da crise da empresa.

In [32]:
# 13. Preço médio (MeanPrice) anual (TradingDate) das 10 ações (NegociationCode) com mais operações na bovespa
top10 = bolsa.groupBy('NegociationCode').count()\
             .orderBy(func.col('count').desc(), func.col('NegociationCode'))\
             .limit(10)
top10.show()
lista_top10 = [r['NegociationCode'] for r in top10.collect()]

bolsa.where(func.col('NegociationCode').isin(lista_top10))\
     .groupBy('ano')\
     .pivot('NegociationCode', lista_top10)\
     .agg(func.round(func.mean('MeanPrice'), 2))\
     .orderBy('ano')\
     .show(40)

+---------------+-----+
|NegociationCode|count|
+---------------+-----+
|         PETR4T|40211|
|         VALE5T|29670|
|         USIM5T|21064|
|         BBAS3T|17628|
|         CSNA3T|15696|
|         GGBR4T|15345|
|         BBDC4T|15321|
|         ITSA4T|12205|
|         CMIG4T|11318|
|         BRKM5T|11078|
+---------------+-----+



+----+--------+--------+--------+-------+--------+-------+--------+-------+-------+--------+
| ano|  PETR4T|  VALE5T|  USIM5T| BBAS3T|  CSNA3T| GGBR4T|  BBDC4T| ITSA4T| CMIG4T|  BRKM5T|
+----+--------+--------+--------+-------+--------+-------+--------+-------+-------+--------+
|1998| 20028.9| 1943.41|    NULL| 944.01| 2523.15|   NULL|  872.02|  71.04|3660.81|    NULL|
|1999|26129.48| 3489.35|  506.06| 697.31| 4042.73|   NULL|  925.79| 108.98|3288.84|    NULL|
|2001| 5357.15| 5310.98|  657.59| 861.21| 5078.37|1885.77| 1195.09| 198.27|2812.86|    NULL|
|2002| 4834.29| 7306.16|  633.64|1063.56| 4283.88|3038.29|  1078.3| 192.87|2885.02|23650.81|
|2003| 5838.46|10003.94|  1663.9|1463.06| 8872.13|3960.49| 1219.13| 246.08|3231.05|34106.06|
|2004| 8550.46|10384.19| 4071.38|2536.29| 8604.14|4850.04|10595.98| 364.02| 5468.6| 7958.87|
|2005| 7806.43|  7267.2| 5067.12|3545.36| 5080.48|3371.54| 8679.61| 592.56|7426.73| 4843.91|
|2006| 4412.33|  5983.6| 7380.99|5354.62| 6552.84|3561.29| 7668.69| 93

In [33]:
# formato longo (uma linha por papel e ano), em SQL
sc.sql("""
    WITH top10 AS (
        SELECT NegociationCode, COUNT(*) AS operacoes
        FROM bovespa
        GROUP BY NegociationCode
        ORDER BY operacoes DESC, NegociationCode
        LIMIT 10
    )
    SELECT b.NegociationCode, b.ano, ROUND(AVG(b.MeanPrice), 2) AS preco_medio
    FROM bovespa b JOIN top10 t ON b.NegociationCode = t.NegociationCode
    GROUP BY b.NegociationCode, b.ano
    ORDER BY b.NegociationCode, b.ano
""").show(12)

+---------------+----+-----------+
|NegociationCode| ano|preco_medio|
+---------------+----+-----------+
|         BBAS3T|1998|     944.01|
|         BBAS3T|1999|     697.31|
|         BBAS3T|2001|     861.21|
|         BBAS3T|2002|    1063.56|
|         BBAS3T|2003|    1463.06|
|         BBAS3T|2004|    2536.29|
|         BBAS3T|2005|    3545.36|
|         BBAS3T|2006|    5354.62|
|         BBAS3T|2007|    4735.87|
|         BBAS3T|2008|    2424.49|
|         BBAS3T|2009|    2251.74|
|         BBAS3T|2010|    3008.12|
+---------------+----+-----------+
only showing top 12 rows



**Resposta.** As 10 ações com mais operações são **PETR4T, VALE5T, USIM5T, BBAS3T, CSNA3T, GGBR4T, BBDC4T, ITSA4T, CMIG4T e BRKM5T**; a tabela dinâmica (pivot) acima mostra o preço médio anual de cada uma. O sufixo **T** indica o **mercado a termo**: esses códigos lideram porque têm vários registros por pregão, um para cada prazo de vencimento (`ForwardMarketTermInDays` de 16, 30, 60, 90, 120 dias...). No mercado à vista, cada papel tem no máximo um registro por pregão e os mais negociados empatam em torno de 5.391 registros, por isso o ranking literal pela contagem de registros é o informativo. Células vazias (`NULL`) são anos em que o papel não foi negociado a termo.

In [34]:
# 14. Desvio Padrão anual do preço médio (MeanPrice) da ação da PETR4 (NegociationCode)
bolsa.where(func.col('NegociationCode') == 'PETR4')\
     .groupBy('ano')\
     .agg(func.round(func.stddev('MeanPrice'), 2).alias('desvio_padrao_MeanPrice'),
          func.count('*').alias('pregoes'))\
     .orderBy('ano')\
     .show(30)

+----+-----------------------+-------+
| ano|desvio_padrao_MeanPrice|pregoes|
+----+-----------------------+-------+
|1998|                5739.29|    197|
|1999|                7692.56|    245|
|2000|               20189.96|    248|
|2001|                 333.46|    246|
|2002|                 613.86|    249|
|2003|                 863.87|    250|
|2004|                 757.82|    249|
|2005|                3377.07|    249|
|2006|                 242.19|    246|
|2007|                1200.45|    245|
|2008|                2311.36|    249|
|2009|                  414.3|    246|
|2010|                 392.23|    247|
|2011|                  304.1|    249|
|2012|                 202.66|    246|
|2013|                 136.15|    248|
|2014|                 319.84|    248|
|2015|                 212.59|    246|
|2016|                 379.52|    249|
|2017|                 131.33|    246|
|2018|                 322.33|    245|
|2019|                 164.43|    248|
+----+-------------------

In [35]:
sc.sql("""
    SELECT ano, ROUND(STDDEV(MeanPrice), 2) AS desvio_padrao_MeanPrice
    FROM bovespa
    WHERE NegociationCode = 'PETR4'
    GROUP BY ano
    ORDER BY ano
""").show(30)

+----+-----------------------+
| ano|desvio_padrao_MeanPrice|
+----+-----------------------+
|1998|                5739.29|
|1999|                7692.56|
|2000|               20189.96|
|2001|                 333.46|
|2002|                 613.86|
|2003|                 863.87|
|2004|                 757.82|
|2005|                3377.07|
|2006|                 242.19|
|2007|                1200.45|
|2008|                2311.36|
|2009|                  414.3|
|2010|                 392.23|
|2011|                  304.1|
|2012|                 202.66|
|2013|                 136.15|
|2014|                 319.84|
|2015|                 212.59|
|2016|                 379.52|
|2017|                 131.33|
|2018|                 322.33|
|2019|                 164.43|
+----+-----------------------+



**Resposta.** Desvio padrão anual do preço médio da PETR4 na tabela acima. O maior valor é o de **2000 (20.189,96)**, inflado pela troca do fator de cotação no meio do ano (lote de mil ações para ação unitária), que mistura duas escalas no mesmo ano. Desconsiderando esse efeito, os anos mais voláteis foram 2005 (3.377,07) e 2008 (2.311,36, crise financeira global); os mais estáveis, 2017 (131,33) e 2013 (136,15).

In [36]:
# 15. Desvio Padrão anual do preço médio (MeanPrice) das 10 ações (NegociationCode) com mais operações na bovespa
bolsa.where(func.col('NegociationCode').isin(lista_top10))\
     .groupBy('ano')\
     .pivot('NegociationCode', lista_top10)\
     .agg(func.round(func.stddev('MeanPrice'), 2))\
     .orderBy('ano')\
     .show(40)

+----+-------+-------+-------+------+-------+-------+-------+------+-------+--------+
| ano| PETR4T| VALE5T| USIM5T|BBAS3T| CSNA3T| GGBR4T| BBDC4T|ITSA4T| CMIG4T|  BRKM5T|
+----+-------+-------+-------+------+-------+-------+-------+------+-------+--------+
|1998|5606.65| 368.32|   NULL| 191.6| 585.16|   NULL| 155.15| 10.38|1204.41|    NULL|
|1999|8365.32|  848.2| 183.43| 73.22|1281.96|   NULL| 158.08| 29.95| 499.99|    NULL|
|2001| 339.67| 273.16| 191.07|101.75|1451.98| 229.54|  90.41| 14.19| 336.19|    NULL|
|2002| 621.85| 1400.2| 116.05|149.92| 568.31| 292.16| 192.49| 28.87| 564.82| 3029.57|
|2003| 897.58|1887.77| 768.66|430.79| 3074.7| 787.98| 109.78| 42.66| 696.65|23881.92|
|2004| 799.22|3965.69| 812.22|419.12|6160.51|  993.5|6370.61| 42.39| 830.88| 2394.26|
|2005| 3576.1| 939.18| 831.13|534.08| 852.93| 974.28| 1963.3|280.94| 1181.2| 4305.43|
|2006| 269.11|2265.75| 709.45|546.05| 511.31| 639.24| 684.06| 69.95| 643.13|  214.69|
|2007|1293.12|1178.07|1799.65|2168.6|2637.78| 668.98|1

In [37]:
sc.sql(f"""
    SELECT NegociationCode, ano, ROUND(STDDEV(MeanPrice), 2) AS desvio_padrao
    FROM bovespa
    WHERE NegociationCode IN ({', '.join(repr(c) for c in lista_top10)})
    GROUP BY NegociationCode, ano
    ORDER BY NegociationCode, ano
""").show(12)

+---------------+----+-------------+
|NegociationCode| ano|desvio_padrao|
+---------------+----+-------------+
|         BBAS3T|1998|        191.6|
|         BBAS3T|1999|        73.22|
|         BBAS3T|2001|       101.75|
|         BBAS3T|2002|       149.92|
|         BBAS3T|2003|       430.79|
|         BBAS3T|2004|       419.12|
|         BBAS3T|2005|       534.08|
|         BBAS3T|2006|       546.05|
|         BBAS3T|2007|       2168.6|
|         BBAS3T|2008|       509.92|
|         BBAS3T|2009|       653.99|
|         BBAS3T|2010|       233.97|
+---------------+----+-------------+
only showing top 12 rows



**Resposta.** Desvio padrão anual do preço médio das 10 ações com mais operações (as mesmas do item 13) na tabela dinâmica acima. Picos isolados, como BRKM5T em 2003 (23.881,92), CSNA3T em 2004 (6.160,51) e BBDC4T em 2004 (6.370,61), coincidem com mudanças de patamar de preço (desdobramentos e grupamentos de ações), e não necessariamente com oscilação de mercado. Em anos recentes os desvios são da ordem de centenas (alguns reais por ação).

In [38]:
# 16. Preço médio (MeanPrice) anual (TradingDate) das ações (NegociationCode) com a maior quantidade de operações de acordo com a moeda (Currency)
from pyspark.sql.window import Window

# 1) papel com mais operações em cada moeda
ranking = bolsa.groupBy('Currency', 'NegociationCode').count()\
               .withColumn('pos', func.row_number().over(
                   Window.partitionBy('Currency').orderBy(func.col('count').desc(), func.col('NegociationCode'))))
lider_moeda = ranking.where(func.col('pos') == 1).drop('pos')
lider_moeda.orderBy(func.col('count').desc()).show()

# 2) preço médio anual desse papel, considerando apenas os pregões naquela moeda
bolsa.join(lider_moeda.select('Currency', 'NegociationCode'), ['Currency', 'NegociationCode'])\
     .groupBy('Currency', 'NegociationCode', 'ano')\
     .agg(func.round(func.mean('MeanPrice'), 2).alias('preco_medio'), func.count('*').alias('operacoes'))\
     .orderBy('Currency', 'ano')\
     .show(40)

+--------+---------------+-----+
|Currency|NegociationCode|count|
+--------+---------------+-----+
|      R$|         PETR4T|40211|
|     CR$|          BBD 4| 2264|
|     CZ$|          PMA 2| 2002|
|    NCZ$|          PET 2|  618|
+--------+---------------+-----+



+--------+---------------+----+-----------+---------+
|Currency|NegociationCode| ano|preco_medio|operacoes|
+--------+---------------+----+-----------+---------+
|     CR$|          BBD 4|1986|    3758.77|      111|
|     CR$|          BBD 4|1990|   131348.1|      391|
|     CR$|          BBD 4|1991|   48068.76|      527|
|     CR$|          BBD 4|1992|   16516.58|      501|
|     CR$|          BBD 4|1993|   62409.34|      492|
|     CR$|          BBD 4|1994|    1313.21|      242|
|     CZ$|          PMA 2|1986|    1718.63|      746|
|     CZ$|          PMA 2|1987|    1507.88|      579|
|     CZ$|          PMA 2|1988|    4754.28|      654|
|     CZ$|          PMA 2|1989|    16335.7|       23|
|    NCZ$|          PET 2|1989|  837994.18|      514|
|    NCZ$|          PET 2|1990| 8839735.72|      104|
|      R$|         PETR4T|1998|    20028.9|      198|
|      R$|         PETR4T|1999|   26129.48|      181|
|      R$|         PETR4T|2001|    5357.15|      827|
|      R$|         PETR4T|20

In [39]:
sc.sql("""
    WITH contagem AS (
        SELECT Currency, NegociationCode, COUNT(*) AS operacoes,
               ROW_NUMBER() OVER (PARTITION BY Currency ORDER BY COUNT(*) DESC, NegociationCode) AS pos
        FROM bovespa
        GROUP BY Currency, NegociationCode
    )
    SELECT b.Currency, b.NegociationCode, b.ano, ROUND(AVG(b.MeanPrice), 2) AS preco_medio
    FROM bovespa b
    JOIN contagem c ON b.Currency = c.Currency AND b.NegociationCode = c.NegociationCode AND c.pos = 1
    GROUP BY b.Currency, b.NegociationCode, b.ano
    ORDER BY b.Currency, b.ano
""").show(40)

+--------+---------------+----+-----------+
|Currency|NegociationCode| ano|preco_medio|
+--------+---------------+----+-----------+
|     CR$|          BBD 4|1986|    3758.77|
|     CR$|          BBD 4|1990|   131348.1|
|     CR$|          BBD 4|1991|   48068.76|
|     CR$|          BBD 4|1992|   16516.58|
|     CR$|          BBD 4|1993|   62409.34|
|     CR$|          BBD 4|1994|    1313.21|
|     CZ$|          PMA 2|1986|    1718.63|
|     CZ$|          PMA 2|1987|    1507.88|
|     CZ$|          PMA 2|1988|    4754.28|
|     CZ$|          PMA 2|1989|    16335.7|
|    NCZ$|          PET 2|1989|  837994.18|
|    NCZ$|          PET 2|1990| 8839735.72|
|      R$|         PETR4T|1998|    20028.9|
|      R$|         PETR4T|1999|   26129.48|
|      R$|         PETR4T|2001|    5357.15|
|      R$|         PETR4T|2002|    4834.29|
|      R$|         PETR4T|2003|    5838.46|
|      R$|         PETR4T|2004|    8550.46|
|      R$|         PETR4T|2005|    7806.43|
|      R$|         PETR4T|2006| 

**Resposta.** O papel com mais operações em cada moeda e o seu preço médio anual (só nos pregões daquela moeda):

- **R$: PETR4T** (40.211 operações), preço médio de 20.028,90 em 1998 a 2.763,94 em 2019 (tabela acima);
- **CR$: BBD 4** (Bradesco PN, 2.264 operações), com médias anuais entre 1.313,21 (1994) e 131.348,10 (1990);
- **CZ$: PMA 2** (Paranapanema, 2.002 operações), de 1.718,63 em 1986 a 16.335,70 em 1989;
- **NCZ$: PET 2** (Petrobras, 618 operações), 837.994,18 em 1989 e 8.839.735,72 em 1990.

Nas moedas antigas o preço médio varia de forma explosiva de um ano para o outro, efeito da inflação e dos cortes de zeros nas trocas de moeda, não de valorização ou desvalorização das empresas.